# 16 FS3 XGBoost Ablation

This notebook is the generator-owned `FS3 XGBoost` ablation workflow for the full combo parent.

Methodology of the upgraded workflow:
- `Stage A`: true top-level ablation on `endogenous_core`, `calendar`, and `exogenous_total`
- `Layer 1`: true mutually exclusive block ablation on the full parent bundle
- `Layer 2`: optional subgroup follow-up for selected Layer 1 blocks
- preflight integrity checks run on the effective branch-specific model columns before results are trusted

Critical validity rule:
- old FS3 combo ablation runs produced before the fixed domestic-versus-neighbor taxonomy, or before the current scheme hashes were stored, are treated as stale and are excluded from the active reporting path

Interpretation rule:
- ablation shows marginal contribution conditional on the rest of the parent bundle staying present
- it does **not** show standalone independent value, and it does **not** prove causal truth


In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys
import time

import pandas as pd
from IPython.display import Markdown, display

NOTEBOOK_CWD = Path.cwd()
REPO_ROOT = next(
    path for path in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents] if (path / "scripts/Data/02_Forecasting/01_DA_prices").exists()
)
os.chdir(REPO_ROOT)
PACKAGE_ROOT = REPO_ROOT / "scripts/Data/02_Forecasting/01_DA_prices"
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.append(str(PACKAGE_ROOT))

from hourly_da.core.config import HourlyDAPipelineConfig
from hourly_da.core.external_features import build_external_family_catalog, load_external_feature_store
from hourly_da.core.methodology import (
    STARTER_ENDOGENOUS_FEATURE_NOTE,
    feature_stage_policy_frame,
    model_status_frame,
    shortlisting_policy_frame,
)
from hourly_da.core.reporting import find_latest_run, load_csv, load_json
from hourly_da.core.tuning import build_tuning_placeholder, tuning_cadence_frame, tuning_snippet_frame
from hourly_da.models import naive_model_names
from hourly_da.notebook_support import estimate_run_duration_seconds, format_duration, load_selected_case_weeks

config = HourlyDAPipelineConfig(
    input_csv=REPO_ROOT / "data/01_cleaned/Day_ahead_prices/DA_prices/hourly/da_prices_all_regions_hourly.csv",
    raw_root=REPO_ROOT / "data/00_Raw/DA_Prices",
    cleaned_feature_root=REPO_ROOT / "data/01_cleaned",
    output_root=REPO_ROOT / "data/02_Forecasting/01_DA_prices/hourly_da",
)
output_root = config.output_root


def latest_run_or_none(run_label: str) -> Path | None:
    try:
        return find_latest_run(output_root, run_label)
    except FileNotFoundError:
        return None


def ensure_required_modules(module_names: list[str], install_command: str | None = None) -> None:
    missing = [module_name for module_name in module_names if importlib.util.find_spec(module_name) is None]
    if not missing:
        return

    message_lines = [
        "Missing required package(s) in the active notebook interpreter: " + ", ".join(missing),
        f"Active interpreter: {sys.executable}",
    ]
    if install_command:
        message_lines.append(f"Install command: {install_command}")
    raise RuntimeError("\n".join(message_lines))


def run_command_with_live_output(command: list[str]) -> None:
    print("Running command:")
    print(" ".join(str(part) for part in command))
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )

    output_tail: list[str] = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        output_tail.append(line)
        if len(output_tail) > 40:
            output_tail.pop(0)

    return_code = process.wait()
    if return_code != 0:
        tail_text = "".join(output_tail).strip()
        message = f"Command failed with exit code {return_code}."
        if tail_text:
            message += "\nLast output:\n" + tail_text
        raise RuntimeError(message)


## Optional execution hook

This snippet controls execution.

What it does:
- reruns the saved `FS3 XGBoost` combo parent if needed
- runs `Stage A` and `Layer 1` by default
- runs `Layer 2` only when you explicitly enable it for selected blocks

How to use it:
- keep `ALLOW_HEAVY_RERUN = False` to read saved artifacts only
- switch `FEATURE_VALUE_STEP` to `all` when the parent itself must be regenerated
- enable only a few Layer 2 targets at a time, because each one adds near-full child reruns


In [ ]:
ALLOW_HEAVY_RERUN = False
FEATURE_VALUE_STEP = "ablation"
RUN_STAGE_A = True
RUN_LAYER_1 = True
RUN_LAYER_2 = False
RUN_DIAGNOSTICS = False

LAYER_2_TARGET_BLOCKS = {
    "domestic_day_ahead_fundamentals": False,
    "neighbor_only_day_ahead_fundamentals": False,
    "domestic_historical_fundamentals": False,
    "neighbor_only_historical_fundamentals": False,
    "engineered_endogenous_history_stats": False,
    "weekly_same_hour_lag_structure": False,
    "structural_capacity": False,
}

selected_layer2_targets = [
    block_name
    for block_name, enabled in LAYER_2_TARGET_BLOCKS.items()
    if bool(enabled)
]

if ALLOW_HEAVY_RERUN:
    estimate = estimate_run_duration_seconds(output_root, "xgboost_fs3_combo_promoted_benchmark")
    if estimate is not None:
        print(
            "Heavy rerun warning: latest comparable parent run "
            f"{estimate['run_id']} suggests about {format_duration(float(estimate['estimate_seconds']))} for one full benchmark pass."
        )
    else:
        print("Heavy rerun warning: no comparable runtime estimate was found for the parent benchmark.")

    ablation_schemes = []
    if RUN_STAGE_A:
        ablation_schemes.append("stage_a_top_level")
    if RUN_LAYER_1:
        ablation_schemes.append("layer1_mutually_exclusive")
    if RUN_LAYER_2 and selected_layer2_targets:
        ablation_schemes.append("layer2_subgroups")

    if not ablation_schemes:
        raise RuntimeError("At least one of RUN_STAGE_A, RUN_LAYER_1, or RUN_LAYER_2 must be enabled.")

    command = [
        sys.executable,
        str(PACKAGE_ROOT / "run_fs3_ordered_benchmarks.py"),
        "--execution-mode",
        "feature_value",
        "--feature-value-step",
        FEATURE_VALUE_STEP,
        "--stage",
        "combo",
        "--feature-value-model-family",
        "xgboost",
        "--ablation-schemes",
        *ablation_schemes,
    ]
    if RUN_LAYER_2 and selected_layer2_targets:
        command.extend(["--layer2-target-block", *selected_layer2_targets])

    started = time.perf_counter()
    run_command_with_live_output(command)
    elapsed_seconds = time.perf_counter() - started
    print(f"Actual wall-clock time: {format_duration(elapsed_seconds)}")
else:
    print("Rerun skipped. Set ALLOW_HEAVY_RERUN = True only when you are ready to execute the finalized pipeline.")
    print(f"Current FEATURE_VALUE_STEP setting: {FEATURE_VALUE_STEP}")
    print(f"Stage A enabled: {RUN_STAGE_A}")
    print(f"Layer 1 enabled: {RUN_LAYER_1}")
    print(f"Layer 2 enabled: {RUN_LAYER_2}")
    print(f"Layer 2 targets: {selected_layer2_targets or 'none'}")
    print(f"Diagnostics enabled: {RUN_DIAGNOSTICS}")


In [ ]:
display(
    feature_stage_policy_frame()
    .loc[lambda df: df["fs_level"] == "FS3"]
    .reset_index(drop=True)
)


In [ ]:
display(
    model_status_frame()
    .loc[lambda df: df["model_family"].isin(['xgboost'])]
    .reset_index(drop=True)
)


In [ ]:
display(pd.DataFrame([build_tuning_placeholder("xgboost", "FS3")]))
display(tuning_snippet_frame(model_family="xgboost", fs_level="FS3"))


In [ ]:
rows = []
for run_label in ['xgboost_fs3_combo_promoted_benchmark']:
    run_dir = latest_run_or_none(run_label)
    rows.append(
        {
            "run_label": run_label,
            "latest_run": str(run_dir) if run_dir is not None else "not yet run",
        }
    )

display(pd.DataFrame(rows))


## Parent reminder and stale-result rule

Read notebook `13` and notebook `14` first when you want the full benchmark comparison against `FS2`.

This notebook answers a narrower follow-up question:
- once the `FS3 XGBoost` combo parent is accepted, which broad blocks still matter when removed one at a time?
- how does that answer change between `D only` and the stitched full horizon?

Why previous numbers can be stale:
- the domestic/crossborder FS3 overlap bug was fixed
- the staged block registry now has explicit scheme hashes and branch-specific preflight checks
- any older saved aggregate without matching hashes is excluded here


In [ ]:
run_dir = latest_run_or_none("xgboost_fs3_combo_promoted_benchmark")
focus_models = ['naive_previous_week', 'naive_previous_year', 'lear_fs2', 'xgboost_fs2', 'xgboost_fs3_combo_promoted']
model_order = {model_name: position for position, model_name in enumerate(focus_models)}

if run_dir is None:
    print("No saved run exists yet for this notebook under the finalized methodology.")
else:
    print(run_dir)
    metrics_by_reporting_level = load_csv(run_dir, "metrics_by_reporting_level.csv")
    timing_summary = load_csv(run_dir, "origin_timing_summary.csv")
    official_naive = load_json(run_dir, "official_naive_reference.json")
    display(
        metrics_by_reporting_level[metrics_by_reporting_level["model"].isin(focus_models)]
        .assign(_model_order=lambda frame: frame["model"].map(model_order).fillna(len(model_order)))
        .sort_values(["dataset_split", "reporting_level_sort_order", "_model_order", "model"])
        .drop(columns=["_model_order"])
        .reset_index(drop=True)
    )
    display(pd.DataFrame([official_naive]))
    display(
        timing_summary[timing_summary["model"].isin(focus_models)]
        .assign(_model_order=lambda frame: frame["model"].map(model_order).fillna(len(model_order)))
        .sort_values(["dataset_split", "_model_order", "model"])
        .drop(columns=["_model_order"])
        .reset_index(drop=True)
    )


## Scheme availability and compatibility

This section checks whether saved staged-ablation artifacts are still compatible with the current code.

What is validated:
- scheme name and version
- effective scheme hash
- feature taxonomy hash
- stored preflight validity

If compatibility breaks, the notebook refuses to treat the old result as active evidence.


In [ ]:
from matplotlib import pyplot as plt
from IPython.display import Markdown

from hourly_da.core.ablation_blocks import (
    SCHEME_NAME_LAYER_1,
    SCHEME_NAME_LAYER_2,
    SCHEME_NAME_STAGE_A,
    supported_layer2_target_blocks,
)
from hourly_da.notebook_support import (
    ablation_effect_label,
    annotate_ablation_metric_slice,
    apply_notebook_display_defaults,
    apply_standard_matplotlib_style,
    compute_fs3_parent_block_diagnostics,
    load_fs3_ablation_bundle,
    plot_feature_family_delta_bars,
    plot_feature_family_heatmap,
    plot_feature_family_origin_stability,
    select_feature_family_metric_slice,
    summarize_fs3_ablation_availability,
)

apply_notebook_display_defaults()
apply_standard_matplotlib_style()

FS3_PARENT_RUN_LABEL = "xgboost_fs3_combo_promoted_benchmark"
FS3_MODEL_FAMILY = "xgboost"
FS3_MODEL_LABEL = "Xgboost"
PRIMARY_SPLIT = "validation"
PRIMARY_METRIC = "mae"
VISIBLE_REPORTING_LEVELS = ("d_only", "stitched_all_horizon")

_fs3_store = load_external_feature_store(config)
_layer2_supported_targets = supported_layer2_target_blocks(_fs3_store.experiment_map())
FS3_PARENT_SPECS = [
    {
        "parent_run_label": FS3_PARENT_RUN_LABEL,
        "model_family": FS3_MODEL_FAMILY,
        "model_label": FS3_MODEL_LABEL,
    }
]
FS3_SCHEME_REQUESTS = [
    {"scheme_name": SCHEME_NAME_STAGE_A, "target_block": None},
    {"scheme_name": SCHEME_NAME_LAYER_1, "target_block": None},
]
FS3_SCHEME_REQUESTS.extend(
    {"scheme_name": SCHEME_NAME_LAYER_2, "target_block": target_block}
    for target_block in _layer2_supported_targets
)

FS3_ABLATION_AVAILABILITY = summarize_fs3_ablation_availability(
    output_root,
    config,
    parent_specs=FS3_PARENT_SPECS,
    scheme_requests=FS3_SCHEME_REQUESTS,
)

FS3_ABLATION_BUNDLES = {}
for request in FS3_SCHEME_REQUESTS:
    key = (str(request["scheme_name"]), str(request["target_block"] or ""))
    bundle = load_fs3_ablation_bundle(
        output_root,
        config,
        parent_run_label=FS3_PARENT_RUN_LABEL,
        model_family=FS3_MODEL_FAMILY,
        scheme_name=str(request["scheme_name"]),
        target_block=str(request["target_block"]) if request["target_block"] else None,
        store=_fs3_store,
    )
    if bundle is not None:
        FS3_ABLATION_BUNDLES[key] = bundle


In [ ]:
availability_view = FS3_ABLATION_AVAILABILITY[
    [
        "model_label",
        "scheme_name",
        "layer_name",
        "target_block",
        "aggregate_run_label",
        "run_id",
        "availability_status",
        "latest_timestamp_label",
        "status_note",
    ]
].rename(
    columns={
        "model_label": "Model",
        "scheme_name": "Scheme",
        "layer_name": "Layer",
        "target_block": "Layer 2 target",
        "aggregate_run_label": "Aggregate run label",
        "run_id": "Run id",
        "availability_status": "Status",
        "latest_timestamp_label": "Latest timestamp",
        "status_note": "Status note",
    }
)
display(availability_view.style.hide(axis="index"))

legacy_rows = FS3_ABLATION_AVAILABILITY[
    FS3_ABLATION_AVAILABILITY["availability_status"].isin(["legacy_incompatible", "invalid", "incomplete"])
].copy()
if not legacy_rows.empty:
    display(
        Markdown(
            "Current compatibility gate: old FS3 combo ablation runs are excluded whenever the saved scheme hash, "
            "feature taxonomy hash, or stored preflight no longer matches the current code. "
            "That includes runs produced before the domestic/crossborder FS3 taxonomy fix."
        )
    )


## Stage A: Top-Level Orientation

What this section does:
- removes one of three top-level blocks from the full `FS3 XGBoost` parent
- checks the group integrity on the effective branch inputs first
- then shows how the child run changes out of sample

Why this step exists:
- it gives a fast high-level orientation before the more detailed Layer 1 block view

How to read it:
- positive delta means the model got worse when the block was removed, so that block was helping
- negative delta means the child improved after removal, so the block may be weak, redundant, or mildly harmful
- day-1 blocks naturally matter more for `D only`; full-horizon deltas can dilute that effect


In [ ]:
bundle = FS3_ABLATION_BUNDLES.get(('stage_a_top_level', ''))

if bundle is None:
    print("No compatible saved bundle is available for this section yet.")
else:
    metadata = bundle["metadata"]
    preflight_summary = bundle["preflight_summary"].copy()
    preflight_block_sizes = bundle["preflight_block_sizes"].copy()
    display(
        pd.DataFrame(
            [
                {
                    "Scheme": metadata.get("ablation_scheme", {}).get("display_name", "stage_a_top_level"),
                    "Layer": metadata.get("layer_name", ""),
                    "Layer 2 target": metadata.get("target_block", ""),
                    "Parent run": metadata.get("parent_run_id", ""),
                    "Aggregate run": metadata.get("run_id", ""),
                    "Scheme hash": metadata.get("scheme_hash", ""),
                    "Feature taxonomy hash": metadata.get("feature_taxonomy_hash", ""),
                    "Preflight valid": bool(metadata.get("preflight_valid", False)),
                }
            ]
        )
    )
    display(preflight_summary)
    display(preflight_block_sizes[["branch_name", "block_name", "block_size", "is_zero_size_block"]])

    if not preflight_summary["valid"].astype(bool).all():
        raise RuntimeError("The stored preflight for this ablation scheme is invalid. Rerun the compatible workflow before interpreting results.")

    stage_candidates = set()
    for reporting_level in VISIBLE_REPORTING_LEVELS:
        metric_slice = select_feature_family_metric_slice(
            bundle["summary"],
            split=PRIMARY_SPLIT,
            reporting_level=reporting_level,
            metric=PRIMARY_METRIC,
        )
        display(Markdown(f"### Validation / {reporting_level.replace('_', ' ').title()}"))
        if metric_slice.empty:
            print("No rows are available for this reporting lens yet.")
            continue

        annotated = annotate_ablation_metric_slice(metric_slice)
        display(
            annotated[
                [
                    "feature_family",
                    "parent_value",
                    "child_value",
                    "delta",
                    "relative_delta",
                    "effect_label",
                ]
            ]
            .rename(
                columns={
                    "feature_family": "Block",
                    "parent_value": "Parent value",
                    "child_value": "Child value",
                    "delta": "Delta",
                    "relative_delta": "Relative delta",
                    "effect_label": "Interpretation",
                }
            )
            .style
            .format(
                {
                    "Parent value": "{:.4f}",
                    "Child value": "{:.4f}",
                    "Delta": "{:+.4f}",
                    "Relative delta": "{:+.3%}",
                }
            )
            .hide(axis="index")
        )

        bucket_rows = []
        for bucket_name in [
            "strong_helpful",
            "mild_helpful",
            "near_zero_uncertain",
            "mild_harmful",
            "strong_harmful",
        ]:
            bucket_blocks = annotated.loc[
                annotated["effect_bucket"].astype(str) == bucket_name,
                "feature_family",
            ].astype(str).tolist()
            bucket_rows.append(
                {
                    "Bucket": ablation_effect_label(bucket_name),
                    "Blocks": ", ".join(bucket_blocks) if bucket_blocks else "-",
                }
            )
            if bucket_name in {"mild_helpful", "near_zero_uncertain", "mild_harmful"}:
                stage_candidates.update(bucket_blocks)
        display(pd.DataFrame(bucket_rows))

        fig = plot_feature_family_delta_bars(annotated)
        if fig is not None:
            display(fig)
            plt.close(fig)

        heatmap_fig = plot_feature_family_heatmap(annotated)
        if heatmap_fig is not None:
            display(heatmap_fig)
            plt.close(heatmap_fig)

        stability_fig = plot_feature_family_origin_stability(
            bundle["by_origin"],
            split=PRIMARY_SPLIT,
            reporting_level=reporting_level,
            metric=PRIMARY_METRIC,
        )
        if stability_fig is not None:
            display(stability_fig)
            plt.close(stability_fig)

    if False:
        display(
            pd.DataFrame(
                [
                    {
                        "Blocks worth optional Layer 2 follow-up": ", ".join(sorted(stage_candidates)) if stage_candidates else "-",
                    }
                ]
            )
        )


## Layer 1: Mutually Exclusive Blocks

What this section does:
- runs the thesis-grade block ablation on a mutually exclusive block registry
- verifies that every effective model column is assigned exactly once before results are shown

Why this step exists:
- it replaces the old overlapping domestic-versus-domestic-plus-neighbors logic
- it gives interpretable blocks that can be discussed in plain language

What conclusions are justified:
- which blocks help this specific `FS3 XGBoost` parent conditional on the rest staying present
- which blocks look weak or uncertain under the current parent tuning

What conclusions are not justified:
- that a block has no standalone predictive value everywhere
- that a weak marginal block is always useless in a different model or feature bundle


In [ ]:
bundle = FS3_ABLATION_BUNDLES.get(('layer1_mutually_exclusive', ''))

if bundle is None:
    print("No compatible saved bundle is available for this section yet.")
else:
    metadata = bundle["metadata"]
    preflight_summary = bundle["preflight_summary"].copy()
    preflight_block_sizes = bundle["preflight_block_sizes"].copy()
    display(
        pd.DataFrame(
            [
                {
                    "Scheme": metadata.get("ablation_scheme", {}).get("display_name", "layer1_mutually_exclusive"),
                    "Layer": metadata.get("layer_name", ""),
                    "Layer 2 target": metadata.get("target_block", ""),
                    "Parent run": metadata.get("parent_run_id", ""),
                    "Aggregate run": metadata.get("run_id", ""),
                    "Scheme hash": metadata.get("scheme_hash", ""),
                    "Feature taxonomy hash": metadata.get("feature_taxonomy_hash", ""),
                    "Preflight valid": bool(metadata.get("preflight_valid", False)),
                }
            ]
        )
    )
    display(preflight_summary)
    display(preflight_block_sizes[["branch_name", "block_name", "block_size", "is_zero_size_block"]])

    if not preflight_summary["valid"].astype(bool).all():
        raise RuntimeError("The stored preflight for this ablation scheme is invalid. Rerun the compatible workflow before interpreting results.")

    stage_candidates = set()
    for reporting_level in VISIBLE_REPORTING_LEVELS:
        metric_slice = select_feature_family_metric_slice(
            bundle["summary"],
            split=PRIMARY_SPLIT,
            reporting_level=reporting_level,
            metric=PRIMARY_METRIC,
        )
        display(Markdown(f"### Validation / {reporting_level.replace('_', ' ').title()}"))
        if metric_slice.empty:
            print("No rows are available for this reporting lens yet.")
            continue

        annotated = annotate_ablation_metric_slice(metric_slice)
        display(
            annotated[
                [
                    "feature_family",
                    "parent_value",
                    "child_value",
                    "delta",
                    "relative_delta",
                    "effect_label",
                ]
            ]
            .rename(
                columns={
                    "feature_family": "Block",
                    "parent_value": "Parent value",
                    "child_value": "Child value",
                    "delta": "Delta",
                    "relative_delta": "Relative delta",
                    "effect_label": "Interpretation",
                }
            )
            .style
            .format(
                {
                    "Parent value": "{:.4f}",
                    "Child value": "{:.4f}",
                    "Delta": "{:+.4f}",
                    "Relative delta": "{:+.3%}",
                }
            )
            .hide(axis="index")
        )

        bucket_rows = []
        for bucket_name in [
            "strong_helpful",
            "mild_helpful",
            "near_zero_uncertain",
            "mild_harmful",
            "strong_harmful",
        ]:
            bucket_blocks = annotated.loc[
                annotated["effect_bucket"].astype(str) == bucket_name,
                "feature_family",
            ].astype(str).tolist()
            bucket_rows.append(
                {
                    "Bucket": ablation_effect_label(bucket_name),
                    "Blocks": ", ".join(bucket_blocks) if bucket_blocks else "-",
                }
            )
            if bucket_name in {"mild_helpful", "near_zero_uncertain", "mild_harmful"}:
                stage_candidates.update(bucket_blocks)
        display(pd.DataFrame(bucket_rows))

        fig = plot_feature_family_delta_bars(annotated)
        if fig is not None:
            display(fig)
            plt.close(fig)

        heatmap_fig = plot_feature_family_heatmap(annotated)
        if heatmap_fig is not None:
            display(heatmap_fig)
            plt.close(heatmap_fig)

        stability_fig = plot_feature_family_origin_stability(
            bundle["by_origin"],
            split=PRIMARY_SPLIT,
            reporting_level=reporting_level,
            metric=PRIMARY_METRIC,
        )
        if stability_fig is not None:
            display(stability_fig)
            plt.close(stability_fig)

    if True:
        display(
            pd.DataFrame(
                [
                    {
                        "Blocks worth optional Layer 2 follow-up": ", ".join(sorted(stage_candidates)) if stage_candidates else "-",
                    }
                ]
            )
        )


## Layer 2: Optional Follow-Up

This section stays selective on purpose.

What it does:
- drills into only the Layer 1 blocks that were explicitly rerun as subgroup analyses
- keeps the full Layer 1 parent bundle fixed in the background

How to interpret it:
- these subgroup deltas answer a local follow-up question inside one chosen Layer 1 block
- they should not be mixed back into Stage A or Layer 1 averages as if they were the same exercise


In [ ]:
available_layer2 = FS3_ABLATION_AVAILABILITY[
    (FS3_ABLATION_AVAILABILITY["scheme_name"] == "layer2_subgroups")
    & (FS3_ABLATION_AVAILABILITY["availability_status"] == "available")
].copy()

if available_layer2.empty:
    print("No compatible Layer 2 subgroup bundles are available yet.")
else:
    for layer2_row in available_layer2.to_dict(orient="records"):
        target_block = str(layer2_row["target_block"])
        bundle = FS3_ABLATION_BUNDLES.get(("layer2_subgroups", target_block))
        if bundle is None:
            continue
        display(Markdown(f"### {target_block.replace('_', ' ').title()}"))
        display(bundle["preflight_summary"])
        metric_rows = []
        for reporting_level in VISIBLE_REPORTING_LEVELS:
            metric_slice = select_feature_family_metric_slice(
                bundle["summary"],
                split=PRIMARY_SPLIT,
                reporting_level=reporting_level,
                metric=PRIMARY_METRIC,
            )
            if metric_slice.empty:
                continue
            annotated = annotate_ablation_metric_slice(metric_slice)
            annotated["reporting_level"] = reporting_level
            metric_rows.append(annotated)
        if not metric_rows:
            print("No validation rows are available for this Layer 2 target.")
            continue
        layer2_table = pd.concat(metric_rows, ignore_index=True)
        display(
            layer2_table[
                [
                    "reporting_level",
                    "feature_family",
                    "parent_value",
                    "child_value",
                    "delta",
                    "relative_delta",
                    "effect_label",
                ]
            ]
            .rename(
                columns={
                    "reporting_level": "Reporting level",
                    "feature_family": "Subgroup",
                    "parent_value": "Parent value",
                    "child_value": "Child value",
                    "delta": "Delta",
                    "relative_delta": "Relative delta",
                    "effect_label": "Interpretation",
                }
            )
            .style
            .format(
                {
                    "Parent value": "{:.4f}",
                    "Child value": "{:.4f}",
                    "Delta": "{:+.4f}",
                    "Relative delta": "{:+.3%}",
                }
            )
            .hide(axis="index")
        )


## Lightweight Diagnostics

What this section does:
- refits the saved parent model once on a representative validation origin
- reports block-level gain and split usage for `XGBoost`

Why this exists:
- it helps explain whether the tree used a block in one representative fit
- it does **not** replace ablation, because a block can appear frequently inside trees while still having weak marginal value once the rest of the parent bundle is present


In [ ]:
if not RUN_DIAGNOSTICS:
    print("Diagnostics are disabled by the notebook config.")
else:
    diagnostics = compute_fs3_parent_block_diagnostics(
        config,
        parent_run_label="xgboost_fs3_combo_promoted_benchmark",
        model_family="xgboost",
        split_name="validation",
        prepared_bundle=None,
    )
    display(diagnostics["reference"])
    if not diagnostics["tables"]:
        print("No lightweight diagnostics were produced for the representative validation fit.")
    else:
        for diagnostic_block in diagnostics["tables"]:
            branch_name = diagnostic_block["branch_name"]
            diagnostic_type = diagnostic_block["diagnostic_type"]
            table = diagnostic_block["table"]
            display(Markdown(f"### {branch_name.replace('_', ' ').title()} / {diagnostic_type.replace('_', ' ').title()}"))
            if diagnostic_type == "lear_coefficients":
                display(
                    table.style
                    .format(
                        {
                            "sum_abs_standardized_coefficient": "{:.4f}",
                        }
                    )
                    .hide(axis="index")
                )
            else:
                display(
                    table.style
                    .format(
                        {
                            "total_gain": "{:.4f}",
                            "total_split_count": "{:.0f}",
                        }
                    )
                    .hide(axis="index")
                )
